# Evaluation Metrics (Qwen judge output)

This notebook computes the requested four metrics from:

- Output: `../output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl`
- Input reference set: `../input/combined_dataset_with_reference_good_row_idx.json`

Metrics:

1. **Perplexity (mu ± std)**
   - Uses `google/embeddinggemma-300m` embeddings.
   - Since embedding models do not provide token-level perplexity, this notebook reports an **embedding-based perplexity proxy**:
     - For selected output text `t` and all reference answers `R`, compute cosine logits `s = cos(t, R) / tau`.
     - Convert to probability of the matched question reference with softmax.
     - `ppl_embed = exp(-log p_match)` (lower is better).

2. **Agreement (mu ± std)**
   - Agreement with human/golden winner (`winner`) after mapping judge outputs to `{model_a, model_b, tie}`.

3. **Consistency (mu ± std)**
   - Multi-run consistency for the same question under same setting.
   - Uses **1-flip consistency** across repeats:
     - `1 - (# winner flips between adjacent repeats / (n-1))`

4. **Error Rate (mu ± std)**
   - Fraction of rows with wrong output format (invalid winner token or invalid single scores).

The final summary is aggregated over condition cells:
`judge_type x prompt_variant x temperature`.

In [10]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)


def resolve_path(candidates: list[str]) -> Path:
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(f"Could not resolve any of: {candidates}")


OUTPUT_PATH = resolve_path([
    "output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl",
    "../output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl",
])

INPUT_PATH = resolve_path([
    "input/combined_dataset_with_reference_good_row_idx.json",
    "../input/combined_dataset_with_reference_good_row_idx.json",
])

print("Output:", OUTPUT_PATH)
print("Input :", INPUT_PATH)


def _last_assistant_text(conv) -> str:
    if isinstance(conv, list):
        for msg in reversed(conv):
            if isinstance(msg, dict) and str(msg.get("role", "")).lower() == "assistant":
                return str(msg.get("content", "")).strip()
    return ""


# Output data (JSONL)
df_out = pd.read_json(OUTPUT_PATH, lines=True)

# Input reference data is JSONL despite .json suffix
df_ref = pd.read_json(INPUT_PATH, lines=True)

df_ref = df_ref.copy()
df_ref["answer_a_text"] = df_ref["conversation_a"].map(_last_assistant_text)
df_ref["answer_b_text"] = df_ref["conversation_b"].map(_last_assistant_text)
df_ref["gold_winner"] = df_ref["winner"]

merge_cols = ["row_idx", "dataset", "gold_winner", "reference_answer", "answer_a_text", "answer_b_text"]
df = df_out.merge(
    df_ref[merge_cols],
    how="left",
    left_on="question_id",
    right_on="row_idx",
)

print("\nLoaded output rows:", len(df_out))
print("Loaded input rows :", len(df_ref))
print("Merged rows       :", len(df))
print("Questions covered :", df["question_id"].nunique())
print("Judge types       :", df["judge_type"].value_counts().to_dict())
print("Prompt variants   :", df["prompt_variant"].value_counts().to_dict())
print("Temperatures      :", sorted(df["temperature"].dropna().unique().tolist()))
print("Repeats           :", sorted(df["repeat_id"].dropna().unique().tolist())[:3], "...", sorted(df["repeat_id"].dropna().unique().tolist())[-3:])
print("Datasets          :", df["dataset"].value_counts(dropna=False).to_dict())

missing_ref = int(df["reference_answer"].isna().sum())
missing_gold = int(df["gold_winner"].isna().sum())
print("Missing reference_answer rows:", missing_ref)
print("Missing gold_winner rows     :", missing_gold)

if missing_ref > 0:
    print("Sample question_id with missing reference:")
    display(df.loc[df["reference_answer"].isna(), ["dataset", "question_id"]].drop_duplicates().head(10))

Output: /home/snt/projects_lujun/LLMJudgeTempCausal/output/test_main_eval_stream_batch_Qwen__Qwen3-Next-80B-A3B-Instruct-FP8.jsonl
Input : /home/snt/projects_lujun/LLMJudgeTempCausal/input/combined_dataset_with_reference_good_row_idx.json

Loaded output rows: 180000
Loaded input rows : 500
Merged rows       : 180000
Questions covered : 500
Judge types       : {'pairwise': 60000, 'single_answer': 60000, 'reference_guided': 60000}
Prompt variants   : {'baseline': 90000, 'cot': 90000}
Temperatures      : [0.01, 0.5, 1.0, 1.5, 2.0, 3.0]
Repeats           : [0, 1, 2] ... [7, 8, 9]
Datasets          : {'mt_bench_human_judgments': 126000, 'mmlu_pro': 54000}
Missing reference_answer rows: 360
Missing gold_winner rows     : 0
Sample question_id with missing reference:


,dataset,question_id
439,mmlu_pro,439


In [11]:
# ---- Normalize judge outputs and compute core metrics (except embedding perplexity) ----

VALID_PAIRWISE = {"A", "B", "C"}


def map_pairwise_to_gold_label(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s == "A":
        return "model_a"
    if s == "B":
        return "model_b"
    if s == "C":
        return "tie"
    return np.nan


# Ensure object dtype columns for mixed string/NaN values
if "predicted_winner" not in df.columns:
    df["predicted_winner"] = pd.Series(index=df.index, dtype="object")
if "format_error" not in df.columns:
    df["format_error"] = pd.Series(index=df.index, dtype="object")


# Pairwise/reference rows: token-based winner
pair_mask = df["judge_type"].isin(["pairwise", "reference_guided"])
df.loc[pair_mask, "winner_token"] = df.loc[pair_mask, "pairwise_winner"].astype(str).str.strip().str.upper()
df.loc[pair_mask, "format_error"] = ~df.loc[pair_mask, "winner_token"].isin(VALID_PAIRWISE)
df.loc[pair_mask, "predicted_winner"] = df.loc[pair_mask, "winner_token"].map(map_pairwise_to_gold_label)


# Single-answer rows: score-based winner
single_mask = df["judge_type"].eq("single_answer")
sa = pd.to_numeric(df.loc[single_mask, "score_a"], errors="coerce")
sb = pd.to_numeric(df.loc[single_mask, "score_b"], errors="coerce")
valid_single = sa.notna() & sb.notna() & sa.between(1, 10) & sb.between(1, 10)

single_pred = pd.Series(
    np.where(sa > sb, "model_a", np.where(sa < sb, "model_b", "tie")),
    index=df.index[single_mask],
    dtype="object",
)
single_pred.loc[~valid_single] = None

df.loc[single_mask, "predicted_winner"] = single_pred
df.loc[single_mask, "format_error"] = ~valid_single


# Safety fill for any unknown type
df["format_error"] = df["format_error"].fillna(True).astype(bool)


# Agreement with gold winner from input file
df["agreement"] = np.where(
    df["predicted_winner"].notna() & df["gold_winner"].notna(),
    (df["predicted_winner"] == df["gold_winner"]).astype(float),
    np.nan,
)


# 1-flip consistency helper

def one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return np.nan
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


cons_rows = []
for (ds, jt, pv, temp, qid), g in df.groupby(
    ["dataset", "judge_type", "prompt_variant", "temperature", "question_id"],
    dropna=False,
):
    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
    cons_rows.append(
        {
            "dataset": ds,
            "judge_type": jt,
            "prompt_variant": pv,
            "temperature": temp,
            "question_id": qid,
            "consistency_1flip": one_flip_consistency(seq),
        }
    )

df_cons = pd.DataFrame(cons_rows)


# Condition-level aggregates used for mu±std summary
cond_keys_with_repeat = ["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"]
cond_keys_no_repeat = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["agreement"]
    .mean()
    .rename("agreement")
    .reset_index()
)

error_by_run = (
    df.groupby(cond_keys_with_repeat, dropna=False)["format_error"]
    .mean()
    .rename("error_rate")
    .reset_index()
)

consistency_by_cond = (
    df_cons.groupby(cond_keys_no_repeat, dropna=False)["consistency_1flip"]
    .mean()
    .rename("consistency")
    .reset_index()
)

print("Format-error overall:", round(df["format_error"].mean() * 100, 2), "%")
print("Agreement rows (non-null):", int(df["agreement"].notna().sum()))
print("Consistency rows (question-level):", int(df_cons["consistency_1flip"].notna().sum()))

print("\nAgreement by run (sample):")
display(agreement_by_run.head(8))

print("Error-rate by run (sample):")
display(error_by_run.head(8))

print("Consistency by condition (sample):")
display(consistency_by_cond.head(8))

Format-error overall: 8.19 %
Agreement rows (non-null): 165259
Consistency rows (question-level): 17828

Agreement by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,agreement
0,mmlu_pro,pairwise,baseline,0.01,0,0.573333
1,mmlu_pro,pairwise,baseline,0.01,1,0.593333
2,mmlu_pro,pairwise,baseline,0.01,2,0.580000
3,mmlu_pro,pairwise,baseline,0.01,3,0.580000
4,mmlu_pro,pairwise,baseline,0.01,4,0.573333
5,mmlu_pro,pairwise,baseline,0.01,5,0.566667
6,mmlu_pro,pairwise,baseline,0.01,6,0.573333
7,mmlu_pro,pairwise,baseline,0.01,7,0.586667


Error-rate by run (sample):


,dataset,judge_type,prompt_variant,temperature,repeat_id,error_rate
0,mmlu_pro,pairwise,baseline,0.01,0,0.0
1,mmlu_pro,pairwise,baseline,0.01,1,0.0
2,mmlu_pro,pairwise,baseline,0.01,2,0.0
3,mmlu_pro,pairwise,baseline,0.01,3,0.0
4,mmlu_pro,pairwise,baseline,0.01,4,0.0
5,mmlu_pro,pairwise,baseline,0.01,5,0.0
6,mmlu_pro,pairwise,baseline,0.01,6,0.0
7,mmlu_pro,pairwise,baseline,0.01,7,0.0


Consistency by condition (sample):


,dataset,judge_type,prompt_variant,temperature,consistency
0,mmlu_pro,pairwise,baseline,0.01,0.970370
1,mmlu_pro,pairwise,baseline,0.50,0.956296
2,mmlu_pro,pairwise,baseline,1.00,0.928148
3,mmlu_pro,pairwise,baseline,1.50,0.887685
4,mmlu_pro,pairwise,baseline,2.00,0.835444
5,mmlu_pro,pairwise,baseline,3.00,0.681201
6,mmlu_pro,pairwise,cot,0.01,0.949630
7,mmlu_pro,pairwise,cot,0.50,0.920838


In [ ]:
# ---- Embedding-based perplexity proxy with google/embeddinggemma-300m ----

MODEL_NAME = "google/embeddinggemma-300m"
TAU = 0.07
BATCH_SIZE = 32


def _l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(n, eps, None)


def load_embedder(model_name: str):
    try:
        from sentence_transformers import SentenceTransformer

        st_model = SentenceTransformer(model_name)

        def encode_texts(texts: list[str]) -> np.ndarray:
            emb = st_model.encode(
                texts,
                batch_size=BATCH_SIZE,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=True,
            )
            return emb.astype(np.float32)

        return encode_texts, "sentence_transformers"
    except Exception as st_err:
        print("sentence_transformers path failed:", repr(st_err))
        print("Falling back to transformers AutoModel mean pooling.")

        import torch
        from transformers import AutoModel, AutoTokenizer

        device = "cuda" if torch.cuda.is_available() else "cpu"
        tok = AutoTokenizer.from_pretrained(model_name)
        mdl = AutoModel.from_pretrained(model_name).to(device)
        mdl.eval()

        def encode_texts(texts: list[str]) -> np.ndarray:
            out = []
            for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Embedding batches"):
                batch = texts[i : i + BATCH_SIZE]
                enc = tok(
                    batch,
                    padding=True,
                    truncation=True,
                    max_length=2048,
                    return_tensors="pt",
                )
                enc = {k: v.to(device) for k, v in enc.items()}
                with torch.no_grad():
                    res = mdl(**enc)
                    if hasattr(res, "last_hidden_state"):
                        hs = res.last_hidden_state
                        mask = enc["attention_mask"].unsqueeze(-1)
                        pooled = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                    elif hasattr(res, "pooler_output") and res.pooler_output is not None:
                        pooled = res.pooler_output
                    else:
                        pooled = res[0][:, 0]
                out.append(pooled.detach().cpu().numpy())
            emb = np.concatenate(out, axis=0).astype(np.float32)
            return _l2_normalize(emb)

        return encode_texts, f"transformers({device})"


encode_texts, backend = load_embedder(MODEL_NAME)
print("Embedding backend:", backend)


qbase = (
    df[["dataset", "question_id", "answer_a_text", "answer_b_text", "reference_answer"]]
    .drop_duplicates(["dataset", "question_id"])
    .sort_values(["dataset", "question_id"])
    .reset_index(drop=True)
)

qbase_valid = qbase[qbase["reference_answer"].notna()].copy().reset_index(drop=True)
print("Questions with valid reference:", len(qbase_valid), "/", len(qbase))

all_texts = pd.unique(
    pd.concat(
        [
            qbase_valid["answer_a_text"].fillna(""),
            qbase_valid["answer_b_text"].fillna(""),
            qbase_valid["reference_answer"].fillna(""),
        ],
        ignore_index=True,
    )
).tolist()

print("Unique texts to embed:", len(all_texts))
vecs = encode_texts(all_texts)
text2vec = {t: vecs[i] for i, t in enumerate(all_texts)}


ref_keys = list(qbase_valid[["dataset", "question_id"]].itertuples(index=False, name=None))
key_to_ref_idx = {k: i for i, k in enumerate(ref_keys)}
ref_matrix = np.vstack([text2vec[t] for t in qbase_valid["reference_answer"].tolist()]).astype(np.float32)


def softmax_prob_of_match(v: np.ndarray, ref_idx: int, tau: float = TAU) -> float:
    logits = (v @ ref_matrix.T) / tau
    z = logits - np.max(logits)
    expz = np.exp(z)
    p = expz[ref_idx] / np.sum(expz)
    return float(np.clip(p, 1e-12, 1.0))


lookup_rows = []
for row in tqdm(qbase_valid.itertuples(index=False), total=len(qbase_valid), desc="Precompute ppl lookup"):
    ds = row.dataset
    qid = int(row.question_id)
    ref_idx = key_to_ref_idx[(ds, qid)]

    va = text2vec.get(row.answer_a_text, None)
    vb = text2vec.get(row.answer_b_text, None)

    if va is not None:
        pa = softmax_prob_of_match(va, ref_idx)
        lookup_rows.append({"dataset": ds, "question_id": qid, "predicted_winner": "model_a", "ppl_embed": math.exp(-math.log(pa))})
    if vb is not None:
        pb = softmax_prob_of_match(vb, ref_idx)
        lookup_rows.append({"dataset": ds, "question_id": qid, "predicted_winner": "model_b", "ppl_embed": math.exp(-math.log(pb))})
    if va is not None and vb is not None:
        vt = va + vb
        vt = vt / max(np.linalg.norm(vt), 1e-12)
        pt = softmax_prob_of_match(vt, ref_idx)
        lookup_rows.append({"dataset": ds, "question_id": qid, "predicted_winner": "tie", "ppl_embed": math.exp(-math.log(pt))})

ppl_lookup = pd.DataFrame(lookup_rows)

df = df.merge(ppl_lookup, how="left", on=["dataset", "question_id", "predicted_winner"])

perplexity_by_run = (
    df.groupby(["dataset", "judge_type", "prompt_variant", "temperature", "repeat_id"], dropna=False)["ppl_embed"]
    .mean()
    .rename("perplexity")
    .reset_index()
)

print("Perplexity rows (non-null):", int(df["ppl_embed"].notna().sum()))
print("Perplexity by run sample:")
display(perplexity_by_run.head(8))

sentence_transformers path failed: ModuleNotFoundError("No module named 'sentence_transformers'")
Falling back to transformers AutoModel mean pooling.
Embedding backend: transformers(cuda)
Questions with valid reference: 499 / 500
Unique texts to embed: 1148


Embedding batches:   0%|          | 0/36 [00:00<?, ?it/s]

Precompute ppl lookup:   0%|          | 0/499 [00:00<?, ?it/s]

Perplexity rows (non-null): 164902
Perplexity by run sample:


,dataset,judge_type,prompt_variant,temperature,repeat_id,perplexity
0,mmlu_pro,pairwise,baseline,0.01,0,14.828345
1,mmlu_pro,pairwise,baseline,0.01,1,14.954242
2,mmlu_pro,pairwise,baseline,0.01,2,14.999430
3,mmlu_pro,pairwise,baseline,0.01,3,14.883019
4,mmlu_pro,pairwise,baseline,0.01,4,14.985002
5,mmlu_pro,pairwise,baseline,0.01,5,14.976889
6,mmlu_pro,pairwise,baseline,0.01,6,14.852217
7,mmlu_pro,pairwise,baseline,0.01,7,14.824428


In [13]:
# ---- Final summary: mu ± std (overall + by dataset) ----

def mu_std(series: pd.Series) -> tuple[float, float]:
    s = pd.to_numeric(series, errors="coerce").dropna()
    return float(s.mean()), float(s.std(ddof=1))


def metric_rows(scope: str, dataset_label: str, p_series: pd.Series, a_series: pd.Series, c_series: pd.Series, e_series: pd.Series) -> list[dict]:
    p_mu, p_std = mu_std(p_series)
    a_mu, a_std = mu_std(a_series)
    c_mu, c_std = mu_std(c_series)
    e_mu, e_std = mu_std(e_series)

    return [
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Perplexity (embedding proxy)",
            "mu": p_mu,
            "std": p_std,
            "mu±std": f"{p_mu:.4f} ± {p_std:.4f}",
            "mu%±std%": np.nan,
        },
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Agreement",
            "mu": a_mu,
            "std": a_std,
            "mu±std": f"{a_mu:.4f} ± {a_std:.4f}",
            "mu%±std%": f"{a_mu*100:.2f}% ± {a_std*100:.2f}%",
        },
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Consistency (1-flip)",
            "mu": c_mu,
            "std": c_std,
            "mu±std": f"{c_mu:.4f} ± {c_std:.4f}",
            "mu%±std%": f"{c_mu*100:.2f}% ± {c_std*100:.2f}%",
        },
        {
            "scope": scope,
            "dataset": dataset_label,
            "Metric": "Error Rate",
            "mu": e_mu,
            "std": e_std,
            "mu±std": f"{e_mu:.4f} ± {e_std:.4f}",
            "mu%±std%": f"{e_mu*100:.2f}% ± {e_std*100:.2f}%",
        },
    ]


summary_rows = []

# Overall
summary_rows += metric_rows(
    scope="overall",
    dataset_label="ALL",
    p_series=perplexity_by_run["perplexity"],
    a_series=agreement_by_run["agreement"],
    c_series=consistency_by_cond["consistency"],
    e_series=error_by_run["error_rate"],
)

# By dataset
for ds in sorted(df["dataset"].dropna().astype(str).unique().tolist()):
    summary_rows += metric_rows(
        scope="by_dataset",
        dataset_label=ds,
        p_series=perplexity_by_run.loc[perplexity_by_run["dataset"] == ds, "perplexity"],
        a_series=agreement_by_run.loc[agreement_by_run["dataset"] == ds, "agreement"],
        c_series=consistency_by_cond.loc[consistency_by_cond["dataset"] == ds, "consistency"],
        e_series=error_by_run.loc[error_by_run["dataset"] == ds, "error_rate"],
    )

summary = pd.DataFrame(summary_rows)

print("Requested metrics (mu ± std):")
display(summary[["scope", "dataset", "Metric", "mu±std", "mu%±std%"]])


# Condition breakdown by dataset
group_cols = ["dataset", "judge_type", "prompt_variant", "temperature"]

agreement_by_cond = (
    agreement_by_run.groupby(group_cols, dropna=False)["agreement"]
    .mean()
    .reset_index()
)

error_by_cond = (
    error_by_run.groupby(group_cols, dropna=False)["error_rate"]
    .mean()
    .reset_index()
)

perplexity_by_cond = (
    perplexity_by_run.groupby(group_cols, dropna=False)["perplexity"]
    .mean()
    .reset_index()
)

cond_table = (
    agreement_by_cond.merge(error_by_cond, on=group_cols, how="outer")
    .merge(consistency_by_cond, on=group_cols, how="outer")
    .merge(perplexity_by_cond, on=group_cols, how="outer")
    .sort_values(["dataset", "judge_type", "prompt_variant", "temperature"])
    .reset_index(drop=True)
)

print("\nCondition breakdown (by dataset):")
display(cond_table)


# Persist results next to output file
summary_path = OUTPUT_PATH.with_name("evaluation_summary_mu_std.csv")
summary_by_dataset_path = OUTPUT_PATH.with_name("evaluation_summary_mu_std_by_dataset.csv")
cond_path = OUTPUT_PATH.with_name("evaluation_by_condition.csv")

summary.to_csv(summary_path, index=False)
summary.to_csv(summary_by_dataset_path, index=False)
cond_table.to_csv(cond_path, index=False)

print("\nSaved:")
print("-", summary_path)
print("-", summary_by_dataset_path)
print("-", cond_path)

Requested metrics (mu ± std):


,scope,dataset,Metric,mu±std,mu%±std%
0,overall,ALL,Perplexity (embedding proxy),106.8772 ± 98.6546,NaN
1,overall,ALL,Agreement,0.5734 ± 0.0604,57.34% ± 6.04%
2,overall,ALL,Consistency (1-flip),0.8274 ± 0.1041,82.74% ± 10.41%
3,overall,ALL,Error Rate,0.0981 ± 0.1435,9.81% ± 14.35%
4,by_dataset,mmlu_pro,Perplexity (embedding proxy),13.6077 ± 1.2582,NaN
5,by_dataset,mmlu_pro,Agreement,0.5882 ± 0.0707,58.82% ± 7.07%
6,by_dataset,mmlu_pro,Consistency (1-flip),0.8061 ± 0.1052,80.61% ± 10.52%
7,by_dataset,mmlu_pro,Error Rate,0.1386 ± 0.1644,13.86% ± 16.44%
8,by_dataset,mt_bench_human_judgments,Perplexity (embedding proxy),200.1467 ± 45.2123,NaN
9,by_dataset,mt_bench_human_judgments,Agreement,0.5586 ± 0.0432,55.86% ± 4.32%



Condition breakdown (by dataset):


,dataset,judge_type,prompt_variant,temperature,agreement,error_rate,consistency,perplexity
0,mmlu_pro,pairwise,baseline,0.01,0.577333,0.000000,0.970370,14.912633
1,mmlu_pro,pairwise,baseline,0.50,0.571333,0.000000,0.956296,14.890680
2,mmlu_pro,pairwise,baseline,1.00,0.581333,0.000000,0.928148,14.888153
3,mmlu_pro,pairwise,baseline,1.50,0.588513,0.006667,0.887685,14.798725
4,mmlu_pro,pairwise,baseline,2.00,0.584712,0.033333,0.835444,14.709879
...,...,...,...,...,...,...,...,...
67,mt_bench_human_judgments,single_answer,cot,0.50,0.538356,0.017143,0.866642,200.066265
68,mt_bench_human_judgments,single_answer,cot,1.00,0.532647,0.021143,0.824003,174.799829
69,mt_bench_human_judgments,single_answer,cot,1.50,0.527873,0.028571,0.782769,175.683333
70,mt_bench_human_judgments,single_answer,cot,2.00,0.523875,0.041429,0.752683,137.871594



Saved:
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/evaluation_summary_mu_std.csv
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/evaluation_summary_mu_std_by_dataset.csv
- /home/snt/projects_lujun/LLMJudgeTempCausal/output/evaluation_by_condition.csv


In [16]:
# ---- Table like manuscript format: Human Bench vs MMLU_PRO ----

import numpy as np
import pandas as pd


def _fmt_mu_std(series: pd.Series, digits: int = 4) -> str:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    mu = float(s.mean())
    std = float(s.std(ddof=1)) if len(s) > 1 else 0.0
    return f"{mu:.{digits}f} ± {std:.{digits}f}"


def _short_model_name(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x)
    s = s.split("/")[-1]
    s = s.replace("-A3B-Instruct-FP8", "")
    s = s.replace("-Instruct", "")
    return s


def _one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return np.nan
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


judge_type_map = {
    "pairwise": "Pairwise Comparison",
    "single_answer": "Single Answer Grading",
    "reference_guided": "Reference Guided Grading",
}
prompt_map = {"baseline": "Direct", "cot": "COT"}


# Run-level stats (mu/std over repeats)
run_group_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "repeat_id"]
agreement_run_model = (
    df.groupby(run_group_cols, dropna=False)["agreement"]
    .mean()
    .rename("agreement")
    .reset_index()
)
error_run_model = (
    df.groupby(run_group_cols, dropna=False)["format_error"]
    .mean()
    .rename("error_rate")
    .reset_index()
)
perplexity_run_model = (
    df.groupby(run_group_cols, dropna=False)["ppl_embed"]
    .mean()
    .rename("perplexity")
    .reset_index()
)


# Question-level 1-flip consistency (mu/std over questions)
cons_records = []
for key, g in df.groupby(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature", "question_id"], dropna=False):
    seq = g.sort_values("repeat_id")["predicted_winner"].dropna().astype(str).tolist()
    cons_records.append(
        {
            "dataset": key[0],
            "judge_model": key[1],
            "judge_type": key[2],
            "prompt_variant": key[3],
            "temperature": key[4],
            "question_id": key[5],
            "consistency_1flip": _one_flip_consistency(seq),
        }
    )
cons_q_model = pd.DataFrame(cons_records)


cond_cols = ["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"]
cond_keys = (
    df[cond_cols]
    .drop_duplicates()
    .sort_values(["dataset", "judge_model", "judge_type", "prompt_variant", "temperature"])
    .reset_index(drop=True)
)

rows = []
for r in cond_keys.itertuples(index=False):
    mask = (
        (agreement_run_model["dataset"] == r.dataset)
        & (agreement_run_model["judge_model"] == r.judge_model)
        & (agreement_run_model["judge_type"] == r.judge_type)
        & (agreement_run_model["prompt_variant"] == r.prompt_variant)
        & (agreement_run_model["temperature"] == r.temperature)
    )

    a_series = agreement_run_model.loc[mask, "agreement"]
    e_series = error_run_model.loc[mask, "error_rate"]
    p_series = perplexity_run_model.loc[mask, "perplexity"]

    c_mask = (
        (cons_q_model["dataset"] == r.dataset)
        & (cons_q_model["judge_model"] == r.judge_model)
        & (cons_q_model["judge_type"] == r.judge_type)
        & (cons_q_model["prompt_variant"] == r.prompt_variant)
        & (cons_q_model["temperature"] == r.temperature)
    )
    c_series = cons_q_model.loc[c_mask, "consistency_1flip"]

    rows.append(
        {
            "dataset": r.dataset,
            "T": r.temperature,
            "Model": _short_model_name(r.judge_model),
            "Judge Type": judge_type_map.get(r.judge_type, r.judge_type),
            "Prompt Strategy": prompt_map.get(r.prompt_variant, r.prompt_variant),
            "Perplexity (µ ± std)": _fmt_mu_std(p_series, digits=2),
            "Aggrement (µ ± std)": _fmt_mu_std(a_series, digits=2),
            "Consistency (µ ± std)": _fmt_mu_std(c_series, digits=2),
            "Error Rate (µ ± std)": _fmt_mu_std(e_series, digits=2),
        }
    )

stats_long = pd.DataFrame(rows)


# Identify the two datasets and map to requested headers
datasets = stats_long["dataset"].dropna().astype(str).unique().tolist()
human_ds = next((d for d in datasets if "mt_bench" in d.lower() or "human" in d.lower()), datasets[0] if datasets else None)
mmlu_ds = next((d for d in datasets if "mmlu" in d.lower()), datasets[1] if len(datasets) > 1 else None)

if human_ds is None or mmlu_ds is None:
    raise ValueError(f"Need two datasets to build this table. Found: {datasets}")

base_cols = ["T", "Model", "Judge Type", "Prompt Strategy"]
metric_cols = [
    "Perplexity (µ ± std)",
    "Aggrement (µ ± std)",
    "Consistency (µ ± std)",
    "Error Rate (µ ± std)",
]

human_part = stats_long.loc[stats_long["dataset"] == human_ds, base_cols + metric_cols].copy()
human_part = human_part.rename(columns={c: f"Human Bench {c}" for c in metric_cols})

mmlu_part = stats_long.loc[stats_long["dataset"] == mmlu_ds, base_cols + metric_cols].copy()
mmlu_part = mmlu_part.rename(columns={c: f"MMLU_PRO {c}" for c in metric_cols})

paper_table = (
    human_part.merge(mmlu_part, on=base_cols, how="outer")
    .sort_values(["T", "Model", "Judge Type", "Prompt Strategy"])
    .reset_index(drop=True)
)

print(f"Human Bench dataset: {human_ds}")
print(f"MMLU_PRO dataset: {mmlu_ds}")
display(paper_table)

# Optional csv export for copy/use in paper
paper_table_path = OUTPUT_PATH.with_name("evaluation_paper_style_table.csv")
paper_table.to_csv(paper_table_path, index=False)
print("Saved:", paper_table_path)

Human Bench dataset: mt_bench_human_judgments
MMLU_PRO dataset: mmlu_pro


,T,Model,Judge Type,Prompt Strategy,Human Bench Perplexity (µ ± std),Human Bench Aggrement (µ ± std),Human Bench Consistency (µ ± std),Human Bench Error Rate (µ ± std),MMLU_PRO Perplexity (µ ± std),MMLU_PRO Aggrement (µ ± std),MMLU_PRO Consistency (µ ± std),MMLU_PRO Error Rate (µ ± std)
0,0.01,Qwen3-Next-80B,Pairwise Comparison,COT,220.31 ± 5.69,0.59 ± 0.01,0.94 ± 0.16,0.04 ± 0.01,14.09 ± 1.14,0.55 ± 0.02,0.95 ± 0.16,0.25 ± 0.03
1,0.01,Qwen3-Next-80B,Pairwise Comparison,Direct,206.29 ± 0.17,0.61 ± 0.00,0.98 ± 0.09,0.00 ± 0.00,14.91 ± 0.07,0.58 ± 0.01,0.97 ± 0.12,0.00 ± 0.00
2,0.01,Qwen3-Next-80B,Reference Guided Grading,COT,215.79 ± 24.21,0.58 ± 0.01,0.90 ± 0.20,0.05 ± 0.01,14.08 ± 0.72,0.64 ± 0.02,0.86 ± 0.22,0.11 ± 0.01
3,0.01,Qwen3-Next-80B,Reference Guided Grading,Direct,204.67 ± 0.69,0.58 ± 0.01,0.94 ± 0.15,0.00 ± 0.00,14.08 ± 0.43,0.65 ± 0.01,0.94 ± 0.16,0.00 ± 0.00
4,0.01,Qwen3-Next-80B,Single Answer Grading,COT,211.55 ± 0.98,0.54 ± 0.01,0.91 ± 0.18,0.01 ± 0.00,13.11 ± 1.41,0.63 ± 0.02,0.83 ± 0.26,0.17 ± 0.02
5,0.01,Qwen3-Next-80B,Single Answer Grading,Direct,210.77 ± 3.54,0.54 ± 0.01,0.95 ± 0.14,0.00 ± 0.00,13.18 ± 0.64,0.70 ± 0.01,0.94 ± 0.16,0.00 ± 0.00
6,0.50,Qwen3-Next-80B,Pairwise Comparison,COT,221.23 ± 8.03,0.59 ± 0.01,0.93 ± 0.17,0.04 ± 0.03,13.43 ± 1.32,0.54 ± 0.03,0.92 ± 0.18,0.26 ± 0.06
7,0.50,Qwen3-Next-80B,Pairwise Comparison,Direct,206.62 ± 0.47,0.61 ± 0.01,0.97 ± 0.12,0.00 ± 0.00,14.89 ± 0.09,0.57 ± 0.01,0.96 ± 0.15,0.00 ± 0.00
8,0.50,Qwen3-Next-80B,Reference Guided Grading,COT,226.74 ± 6.48,0.57 ± 0.01,0.88 ± 0.21,0.06 ± 0.03,13.70 ± 1.00,0.62 ± 0.03,0.85 ± 0.23,0.13 ± 0.05
9,0.50,Qwen3-Next-80B,Reference Guided Grading,Direct,204.92 ± 0.73,0.57 ± 0.01,0.92 ± 0.18,0.00 ± 0.00,13.51 ± 0.69,0.64 ± 0.01,0.90 ± 0.21,0.00 ± 0.00


Saved: /home/snt/projects_lujun/LLMJudgeTempCausal/output/evaluation_paper_style_table.csv
